Random Forest Qualification

In [1]:
import fastf1
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sys import prefix
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import fastf1
fastf1.Cache.enable_cache('../f1_cache')

In [4]:
# 1. LOAD DATA
quali_df = pd.read_csv('../archive/qualifying.csv', na_values=r'\N')
races_df = pd.read_csv('../archive/races.csv', na_values=r'\N')
results_df = pd.read_csv('../archive/results.csv', na_values=r'\N')

# 2. TARGET VARIABLE: SESSION REACHED (1, 2, or 3)
def get_session_reached(pos):
    if pos <= 10: return 3  
    if pos <= 15: return 2  
    return 1          

quali_df['QualiSession'] = quali_df['position'].apply(get_session_reached)

# 3. BASE MERGE & CHRONOLOGICAL SORT
full_quali = pd.merge(quali_df, races_df[['raceId', 'year', 'date', 'circuitId']], on='raceId')
full_quali = full_quali.sort_values('date')

# 4. FEATURE: TRACK EXPERIENCE
full_quali['TrackExperience'] = full_quali.groupby(['driverId', 'circuitId']).cumcount()

# 5. FEATURE: STRATEGIC EXIT (GRID PENALTY)
penalty_check = pd.merge(
    full_quali, 
    results_df[['raceId', 'driverId', 'grid']], 
    on=['raceId', 'driverId'],
    how='left'
)
penalty_check['GridPenalty'] = (penalty_check['grid'] - penalty_check['position'] >= 10).astype(int)

# --- CALCULATE HISTORICAL MEMORY ON THE FULL TIMELINE ---

# 6. FEATURE: DRIVER FORM (Rolling 3-race avg)
penalty_check['DriverForm'] = penalty_check.groupby('driverId')['position'].transform(
    lambda x: x.shift(1).rolling(window=3).mean()
)

penalty_check['TrackForm'] = penalty_check.groupby(['driverId','circuitId'])['position'].transform(
    lambda x: x.shift(1).rolling(window=3).mean()
)

penalty_check['TrackForm'] = penalty_check['TrackForm'].fillna(penalty_check['DriverForm'])

# 7. FEATURE: CONSTRUCTOR FORM (Team's rolling 3-race avg)
team_race_avg = penalty_check.groupby(['raceId', 'constructorId'])['position'].mean().reset_index()
team_race_avg.rename(columns={'position': 'Team_Race_Avg'}, inplace=True)

team_race_avg['ConstructorForm'] = team_race_avg.groupby('constructorId')['Team_Race_Avg'].transform(
    lambda x: x.shift(1).rolling(window=3).mean()
)

penalty_check = pd.merge(penalty_check, team_race_avg[['raceId', 'constructorId', 'ConstructorForm']], 
                         on=['raceId', 'constructorId'], how='left')

# 8. FEATURE: TEAMMATE DELTA
# Trick: Team Total Position - My Position = Teammate's Position
team_pos_sum = penalty_check.groupby(['raceId', 'constructorId'])['position'].transform('sum')
penalty_check['TeammatePosition'] = team_pos_sum - penalty_check['position']
# Negative delta means the driver qualified better than their teammate
penalty_check['TeammateDelta'] = penalty_check['position'] - penalty_check['TeammatePosition']

# --- NOW SLICE THE DATASET FOR THE ML MODEL ---

# 9. CREATE THE FINAL ML DATASET (2018+)
dataset = penalty_check[penalty_check['year'] >= 2018].copy()

# Drop rows where the rolling averages created NaNs (drivers/teams in their first 3 races)
dataset = dataset.dropna(subset=['DriverForm', 'ConstructorForm'])

print(f"Dataset prepared with {len(dataset)} rows.")
display(dataset.head(10))

Dataset prepared with 3357 rows.


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,QualiSession,...,date,circuitId,TrackExperience,grid,GridPenalty,DriverForm,TrackForm,ConstructorForm,TeammatePosition,TeammateDelta
7518,7551,989,838,1,2,12,1:24.073,1:23.853,NaN,2,...,2018-03-25,1,1,11.0,0,13.666667,13.666667,12.166667,11,1
7519,7559,989,842,5,10,20,1:25.295,NaN,NaN,1,...,2018-03-25,1,0,20.0,0,17.000000,17.000000,15.833333,16,4
7522,7556,989,828,15,9,17,1:24.556,NaN,NaN,1,...,2018-03-25,1,4,17.0,0,18.000000,15.666667,17.500000,18,-1
7523,7555,989,843,5,28,16,1:24.532,NaN,NaN,1,...,2018-03-25,1,0,16.0,0,16.000000,16.000000,15.833333,20,-4
7524,7554,989,839,10,31,15,1:24.503,1:24.786,NaN,2,...,2018-03-25,1,1,14.0,0,8.666667,8.666667,8.333333,13,2
7525,7553,989,840,3,18,14,1:24.464,1:24.230,NaN,2,...,2018-03-25,1,1,13.0,0,15.000000,15.000000,12.666667,19,-5
7526,7552,989,815,10,11,13,1:24.344,1:24.005,NaN,2,...,2018-03-25,1,7,12.0,0,8.000000,11.666667,8.333333,15,-2
7527,7550,989,4,1,14,11,1:23.597,1:23.692,NaN,2,...,2018-03-25,1,14,10.0,0,10.666667,10.000000,12.166667,12,-1
7528,7548,989,832,4,55,9,1:23.529,1:23.061,1:23.577,3,...,2018-03-25,1,3,9.0,0,10.000000,7.666667,8.833333,8,1
7529,7541,989,8,6,7,2,1:23.096,1:22.507,1:21.828,3,...,2018-03-25,1,13,2.0,0,4.333333,4.333333,3.166667,3,-1


In [ ]:
encoded_data = pd.get_dummies(dataset, columns=['constructorId', 'circuitId'], prefix=['Team', 'Track'])


train_data = encoded_data[encoded_data['year'] <= 2023]
test_data = encoded_data[encoded_data['year'] > 2023]
print(f"Training rows: {len(train_data)}")
print(f"Testing rows: {len(test_data)}")

encoded_cols = [col for col in train_data.columns if col.startswith('Team_') or col.startswith('Track_')]
feature_cols = ['ConstructorForm', 'TrackExperience', 'TeammateDelta', 'GridPenalty', 'DriverForm','TrackForm'] + encoded_cols
X_train = train_data[feature_cols]
Y_train = train_data['QualiSession']
X_test = test_data[feature_cols]
Y_test = test_data['QualiSession']

#Train model
model = LogisticRegression(class_weight='balanced',max_iter=10000)
model.fit(X_train, Y_train)
predictions = model.predict(X_test)

print(f"LR Accuracy: {accuracy_score(Y_test, predictions) * 100:.2f}%")
print("\n--- LR CONFUSION MATRIX ---")
print(confusion_matrix(Y_test, predictions))

model2 = RandomForestClassifier(random_state=42,n_estimators=500,max_depth=15,min_samples_split=5)
model2.fit(X_train,Y_train)
predictions2 = model2.predict(X_test)
print(f"RF Accuracy: {accuracy_score(Y_test, predictions2) * 100:.2f}%")
print("\n--- RF CONFUSION MATRIX ---")
print(confusion_matrix(Y_test, predictions2))


Training rows: 2423
Testing rows: 934
LR Accuracy: 68.74%

--- LR CONFUSION MATRIX ---
[[140  80   8]
 [ 67 119  47]
 [ 11  79 383]]
RF Accuracy: 68.42%

--- RF CONFUSION MATRIX ---
[[144  54  30]
 [ 73  81  79]
 [ 13  46 414]]
